# Week 9: RAG: retrieval-augmented generation

LLMs make things up when they don't know (a **hallucination**). RAG fixes some of that: first **retrieve** the passages that answer a question, then ask the LLM to answer **only from those passages**, and cite them.

This week you also pick your capstone brief.

**Before you start:** add the course document dataset with **Add input** (search *akada-labs-course-data*), or upload your own `.txt` file and change `DOC_PATH`. Turn **Internet on** so the embedding model can download. Also add your API key as a secret. On Kaggle: **Add-ons > Secrets**, name it `GROQ_API_KEY` (or `GEMINI_API_KEY`), and switch it on for this notebook. Turn **Internet on** in the settings panel. On Colab: click the key icon on the left and add the same name.

Run each cell with **Shift + Enter**. Every exercise has a check cell that prints **PASS** or a hint.
At the end you get a completion code for the submission form.

In [ ]:
# Run this cell first. You don't need to read it: it holds the self-checks.
EMAIL = ""  # <- the email you applied with

import hashlib, json, os, re

_passed = set()

def ok(n):
    _passed.add(n)
    print(f"PASS: exercise {n} done.")

def hint(message):
    print(f"Not yet. Hint: {message}")

def free_text(n, text, min_sentences):
    parts = [p for p in re.split(r"[.!?]+", text or "") if len(p.split()) >= 3]
    if len(parts) >= min_sentences:
        print("Saved. A TA will read this answer.")
        ok(n)
    else:
        hint(f"write at least {min_sentences} full sentences inside the quotes.")

def completion_code(week, exercises):
    if "@" not in EMAIL:
        return print("Put your email in the first cell, run it, then run this cell again.")
    missing = sorted(set(exercises) - _passed)
    if missing:
        return print(f"Almost there. Not passed yet: {missing}. Run those exercises and checks again.")
    digest = hashlib.sha256(f"akada-labs-cohort-1|{week}|{EMAIL.strip().lower()}".encode()).hexdigest()
    print(f"Well done! Your completion code is: AK-W{week:02d}-{digest[:6].upper()}")
    print("Paste it into the lab submission form.")

def get_secret(name):
    """Read an API key from Kaggle or Colab secrets. Never paste keys into a cell."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    raise RuntimeError(
        f"No secret called {name}. On Kaggle: Add-ons > Secrets. On Colab: the key icon on the left. "
        "Add it there, switch it on for this notebook, then run this cell again.")

print("Ready.")

In [ ]:
!pip install -q openai sentence-transformers

In [ ]:
# Pick one provider. Both have a free tier.
PROVIDER = "groq"   # or "gemini"

PROVIDERS = {
    # name: (secret name, OpenAI-compatible endpoint, model)
    "groq":   ("GROQ_API_KEY",   "https://api.groq.com/openai/v1", "llama-3.1-8b-instant"),
    "gemini": ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.5-flash"),
}
# Model names change. If you get a "model not found" error, check the provider's model list
# (console.groq.com/docs/models or ai.google.dev/gemini-api/docs/models) and update MODEL.

from openai import OpenAI

SECRET_NAME, BASE_URL, MODEL = PROVIDERS[PROVIDER]
client = OpenAI(api_key=get_secret(SECRET_NAME), base_url=BASE_URL)
print("Using", PROVIDER, MODEL)

import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
DOC_PATH = Path("/kaggle/input/akada-labs-course-data/document.txt")
text = DOC_PATH.read_text(encoding="utf-8")
print(len(text.split()), "words")

## Exercise 1: Chunk the document

Write `chunk(text, size=150, overlap=30)` that splits the text into chunks of `size` words, each starting `size - overlap` words after the last, so neighbouring chunks share some words. Store the result in `chunks`.

In [ ]:
def chunk(text, size=150, overlap=30):
    pass

chunks = chunk(text)
print(len(chunks) if chunks else chunks, "chunks")

In [ ]:
try:
    n_words = len(text.split())
    expected = len(range(0, n_words, 120))
    sizes = [len(c.split()) for c in chunks]
    if not (abs(len(chunks) - expected) <= 1 and max(sizes) <= 150 and min(sizes[:-1]) >= 140):
        hint(f"expected about {expected} chunks of 150 words. Step through the words with range(0, len(words), size - overlap).")
    else:
        ok(1)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 2: Build an index

An **index** here is just the chunks plus their embeddings. Embed every chunk into `index_vectors` (use `normalize_embeddings=True` so a dot product gives cosine similarity).

In [ ]:
index_vectors = None

print(None if index_vectors is None else index_vectors.shape)

In [ ]:
try:
    if len(index_vectors) != len(chunks):
        hint("embed every chunk: embedder.encode(chunks, normalize_embeddings=True)")
    else:
        ok(2)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 3: Retrieve

Write `retrieve(question, k=3)` that returns a list of `(chunk_number, chunk_text)` for the `k` best chunks.

In [ ]:
def retrieve(question, k=3):
    pass

for number, passage in retrieve("What rights does a data subject have?") or []:
    print(number, passage[:150])

In [ ]:
try:
    # Course team: update if you change the document
    hits = retrieve("How quickly must a data breach be reported to the regulator?")
    if len(hits) == 3 and any("72 hours" in p for _, p in hits):
        ok(3)
    else:
        hint("return the 3 best (number, text) pairs, most similar first.")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 4: Answer with citations

Write `answer_question(question)`. It should retrieve 3 chunks, put them in the prompt labelled like `[chunk 12]`, and tell the model to answer only from them, citing the chunk numbers it used, and to say *"I don't know"* if the answer isn't there. Return `(answer_text, chunk_numbers)`.

In [ ]:
def answer_question(question):
    hits = retrieve(question)
    context = "\n\n".join(f"[chunk {n}] {p}" for n, p in hits)
    system = ""   # your instructions
    # call the model, then return its answer and the chunk numbers you retrieved
    pass

reply, used = answer_question("What must a company do after a data breach?")
print(reply)

In [ ]:
try:
    cited = {int(n) for n in re.findall(r"chunk\s*(\d+)", reply, flags=re.I)}
    if cited & set(used):
        ok(4)
    else:
        hint("the answer must cite at least one retrieved chunk, like [chunk 12]. Say so in the system prompt.")
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Exercise 5: Break it

Find a question your RAG bot gets wrong (try something vague, in Pidgin, or about a topic not in the document). In `answer`, write the question, what went wrong, and why: was retrieval or generation to blame?

In [ ]:
bad_question = ""
print(answer_question(bad_question)[0] if bad_question else "")

answer = """

"""

In [ ]:
try:
    free_text(5, answer, 2)
except Exception as error:
    hint(f"something above isn't finished yet ({type(error).__name__}: {error}). Run the cells in order.")

## Pick your capstone

Choose one brief (or propose your own) and post it in the cohort WhatsApp group by Saturday:

| Brief | What it does | Builds on |
| --- | --- | --- |
| Pidgin sentiment checker | Labels customer messages as positive, negative or neutral, in Pidgin or English | Weeks 5 and 7 |
| Market price agent | Answers "how has rice price changed in Kano?" by calling a price tool | Weeks 3 and 10 |
| Ask-the-law bot | Answers questions about a Nigerian document, with citations | Weeks 8 and 9 |
| Bank alert parser | Turns pasted SMS alerts into a spending table | Week 7 |

## Finish

Run the cell below. If every exercise passed, it prints your completion code.

In [ ]:
completion_code(9, [1, 2, 3, 4, 5])